In  a docker terminal run
jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root

then open jupyter in the printed link (such as http://127.0.0.1:8888/tree?token=4d9d578397aede50fc5bd2d92794b562a6bbcbc73b2dfe51)

CODE HERE, REFRESH AND EXECUTE IN THE BROWSER

In [1]:
import os
import sys
import django

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

sys.path.append('/app')

os.environ['DJANGO_SETTINGS_MODULE'] = 'app.settings'
os.environ['PYTHONPATH'] = '/app'

django.setup()

In [2]:
import os
import json
import xarray as xr
import netCDF4 as nc
import numpy as np
import rasterio
from datetime import datetime, timedelta
from app import settings

In [3]:
clc = rasterio.open('./monica_geodata/1000mx1000m/clc_buek_4326_gen_clc_1000m.tif')
buek = rasterio.open('./monica_geodata/1000mx1000m/clc_buek_4326_gen_1000m.tif')

profile = buek.profile.copy()
clc = rasterio.open('./monica_geodata/1000mx1000m/clc_buek_4326_gen_clc_1000m.tif')
height, width = clc.shape
nodata = clc.nodata
profile = clc.profile.copy()
profile.update(dtype=rasterio.int16, count=1, compress='deflate')

In [4]:
np.unique(clc.read(1))

array([-9999,    12,    13,    21,    22,    23,    31,    33,    51,
         111,   112,   121,   122,   123,   124,   131,   132,   133,
         231,   242,   243,   311,   312,   313,   321,   322,   324,
         331,   412], dtype=int16)

In [5]:
profile

{'driver': 'GTiff', 'dtype': 'int16', 'nodata': -9999.0, 'width': 654, 'height': 866, 'count': 1, 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'), 'transform': Affine(0.014029916273700307, 0.0, 5.866250515,
       0.0, -0.008991228240184754, 55.056526184), 'blockxsize': 654, 'blockysize': 6, 'tiled': False, 'compress': 'deflate', 'interleave': 'band'}

In [6]:
mask = clc.read(1) == 21
buek_masked = np.where(mask, buek.read(1), clc.nodata).astype(profile['dtype'])


In [26]:
# write the new masked buek_id.tif
with rasterio.open('./monica_geodata/1000mx1000m/buek_id_agriculture_masked_4326.tif', 'w', driver='GTiff',
                   height=buek_masked.shape[0], 
                   width=buek_masked.shape[1],
                   count=1, 
                   dtype=buek_masked.dtype,
                   crs=clc.crs, 
                   nodata=clc.nodata,
                   transform=clc.transform) as dst:
    #dst.write(buek_masked, 1)
    print(height, width)

866 654


In [ ]:
forest_mask = np.isin(clc.read(1), [31, 311, 312, 313])

In [ ]:
"""
The following code creates maps according for sowing and harvest dates.
"""

159010

In [3]:
from monica import models
from collections import defaultdict
from math import sqrt
from django.db import models as d_models


In [8]:
grid_points = models.DWDGridToPointIndices.objects.filter(is_valid=True)

In [9]:
cultivar_ids = models.SeedHarvestDates.objects.values_list('cultivar_parameters', flat=True).distinct()

In [10]:
cultivar_ids

<QuerySet [1, 4, 11, 16, 17, 18, 22, 26, 27, 32, 37, 57, None]>

In [16]:
for cultivar in cultivar_ids:
    

<QuerySet [43204, 43205, 43206, 43207, 43208, 43209, 43210, 43211, 43212, 43213, 43214, 43215, 43216, 43217, 43218, 43219, 43220, 43221, 43222, 43223, '...(remaining elements truncated)...']>

In [36]:

for c_id in cultivar_ids:
    if c_id in (18, 4, 16, 26, 27, 17, 22, 37, 32, 11, 57, None):
        cultivar_stations = models.SeedHarvestDates.objects.filter(cultivar_parameters__id=c_id)
        stations = list(cultivar_stations.values('climate_station_id', 'lat', 'lon'))
        new_grid = np.full((height, width), nodata, dtype=np.int32)
         # iterate over each grid cell
        for gp in grid_points:
            min_dist = float("inf")
            nearest_station_id = nodata
    
            for station in stations:
                d = sqrt((gp.lat - station['lat'])**2 + (gp.lon - station['lon'])**2)
                if d < min_dist:
                    min_dist = d
                    nearest_station_id = station['climate_station_id']
    
            # write result into raster grid
            new_grid[gp.lat_idx, gp.lon_idx] = nearest_station_id
    
        with rasterio.open(
            f'nearest_station_cultivar_{c_id}.tif',
            'w',
            driver='GTiff',
            height=height,
            width=width,
            count=1,
            dtype=new_grid.dtype,
            crs=clc.crs,
            transform=clc.transform,
            nodata=nodata,
            compress='deflate'
        ) as dst:
            dst.write(new_grid, 1)
    
        print(f"  → finished cultivar {c_id}")
        
        

  → finished cultivar 18
  → finished cultivar 4
  → finished cultivar 16
  → finished cultivar 26
  → finished cultivar 27
  → finished cultivar 17
  → finished cultivar 22
  → finished cultivar 37
  → finished cultivar 32
  → finished cultivar 11
  → finished cultivar 57
  → finished cultivar None


In [6]:
def doy_to_iso(doy):
        if doy is None:
            return None
        date = datetime(2001, 1, 1) + timedelta(days=doy - 1)  # non-leap year
        # TODO: check if the first ratation is always 0001
        return f"0001-{date.strftime('%m-%d')}"
    # cultivar is winter wheat!


In [23]:
cultivar_name = 'winter wheat'

cultivar = models.CultivarParameters.objects.get(name=cultivar_name)
sowing_dates_list = models.SeedHarvestDates.objects.filter(cultivar_parameters=cultivar).values('climate_station__id', 'avg_sowing_doy', 'avg_harvest_doy')

sowing_dates_list
sowing_dates_per_station = {data['climate_station__id']: {'sowing_date': doy_to_iso(data['avg_sowing_doy']), 'harvest_date': doy_to_iso(data['avg_harvest_doy'])} for data in sowing_dates_list}
print("Sowing dates loaded", sowing_dates_per_station)

Sowing dates loaded {3: {'sowing_date': '0000-10-05', 'harvest_date': '0000-08-03'}, 98: {'sowing_date': '0000-09-29', 'harvest_date': '0000-08-12'}, 140: {'sowing_date': '0000-09-28', 'harvest_date': '0000-07-30'}, 164: {'sowing_date': '0000-09-21', 'harvest_date': '0000-07-20'}, 183: {'sowing_date': '0000-09-27', 'harvest_date': '0000-08-11'}, 198: {'sowing_date': '0000-10-10', 'harvest_date': '0000-07-26'}, 222: {'sowing_date': '0000-10-03', 'harvest_date': '0000-08-11'}, 282: {'sowing_date': '0000-10-07', 'harvest_date': '0000-07-30'}, 501: {'sowing_date': '0000-10-04', 'harvest_date': '0000-08-16'}, 524: {'sowing_date': '0000-10-15', 'harvest_date': '0000-08-08'}, 591: {'sowing_date': '0000-09-26', 'harvest_date': '0000-08-09'}, 596: {'sowing_date': '0000-09-25', 'harvest_date': '0000-08-06'}, 640: {'sowing_date': '0000-10-02', 'harvest_date': '0000-08-15'}, 662: {'sowing_date': '0000-10-07', 'harvest_date': '0000-08-02'}, 717: {'sowing_date': '0000-10-07', 'harvest_date': '0000-0

In [24]:
sowing_dates_per_station[3]

{'sowing_date': '0000-10-05', 'harvest_date': '0000-08-03'}

In [8]:
from django.db.models import Min, Max

min_sowing_date = models.SeedHarvestDates.objects.filter(cultivar_parameters=cultivar).aggregate(Min('avg_sowing_doy'))['avg_sowing_doy__min']
max_harvest_date = models.SeedHarvestDates.objects.filter(cultivar_parameters=cultivar).aggregate(Max('avg_harvest_doy'))['avg_harvest_doy__max']


In [9]:
crp = models.CropResidueParameters.objects.get(species_parameters=cultivar.species_parameters, is_default=True)
crp

<CropResidueParameters: CropResidueParameters object (68)>

In [10]:
agri_buek = rasterio.open(os.path.join(settings.BASE_DIR, 'monica', 'monica_geodata', '1000mx1000m', 'buek_id_agriculture_masked_4326.tif'))


In [11]:
arr = agri_buek.read(1)

In [12]:
arr.shape

(866, 654)

In [60]:
from monica import views
from buek import models as buek_models
import requests

In [14]:
climate_stations_tif = rasterio.open(os.path.join(settings.BASE_DIR, 'monica', 'monica_geodata', '1000mx1000m', 'nearest_station_per_cultivar', f'nearest_station_cultivar_{cultivar.id}.tif'))
climate_stations_arr = climate_stations_tif.read(1)

In [16]:
no = climate_stations_arr[111,200]

In [17]:
int(no)

23

In [61]:
views.monica_run_over_germany()

2734 unique buek ids
Soil profiles loaded
sowing dates loaded
-9999 climate_stations_arr[i, j]


KeyError: -9999

In [14]:
climate_stations_tif = rasterio.open(os.path.join(settings.BASE_DIR, 'monica', 'monica_geodata', '1000mx1000m', 'nearest_station_per_cultivar', f'nearest_station_cultivar_{cultivar.id}.tif'))


NameError: name 'cultivar' is not defined

In [59]:
cultivar_name = 'winter wheat'
    # TODO: check with Claas or Marlene what parameters to use!
simulation_settings = models.UserSimulationSettings.objects.get(is_default=True).to_json()
user_crop_parameters = models.UserCropParameters.objects.get(is_default=True).to_json()
user_environment_parameters = models.UserEnvironmentParameters.objects.get(is_default=True).to_json()
user_soil_moisture_parameters = models.UserSoilMoistureParameters.objects.get(is_default=True).to_json()
user_soil_temperature_parameters = models.SoilTemperatureModuleParameters.objects.get(is_default=True).to_json()
user_soil_transport_parameters = models.UserSoilTransportParameters.objects.get(is_default=True).to_json()
user_soil_organic_parameters = models.UserSoilOrganicParameters.objects.get(is_default=True).to_json()

print('Done getting parameters')

# load all relevant soil data for Germany- otherwise the query on each pixel would take too long
agri_buek = rasterio.open(os.path.join(settings.BASE_DIR, 'monica', 'monica_geodata', '1000mx1000m', 'buek_id_agriculture_masked_4326.tif'))
unique_buek_ids = np.unique(agri_buek.read(1))
unique_buek_ids = unique_buek_ids[unique_buek_ids != -9999]

soil_profiles = buek_models.SoilProfile.objects.filter(id__in=unique_buek_ids)
print(len(soil_profiles), "unique buek ids")

soil_profile_dict = {sp.id: sp.get_monica_horizons_json()[0] for sp in soil_profiles}
print("Soil profiles loaded")

def doy_to_iso(doy):
    if doy is None:
        return None
    date = datetime(2001, 1, 1) + timedelta(days=doy - 1)  # non-leap year
    # TODO: check if the first ratation is always 0000
    return f"0000-{date.strftime('%m-%d')}"
# cultivar is winter wheat!
cultivar = models.CultivarParameters.objects.get(name=cultivar_name)

min_sowing_date = models.SeedHarvestDates.objects.filter(cultivar_parameters=cultivar).aggregate(Min('avg_sowing_doy'))['avg_sowing_doy__min']
max_harvest_date = models.SeedHarvestDates.objects.filter(cultivar_parameters=cultivar).aggregate(Max('avg_harvest_doy'))['avg_harvest_doy__max']
sowing_dates_list = models.SeedHarvestDates.objects.filter(cultivar_parameters=cultivar).values('climate_station__id', 'avg_sowing_doy', 'avg_harvest_doy')

print(sowing_dates_list)
sowing_dates_per_station = {data['climate_station__id']: {'sowing_date': doy_to_iso(data['avg_sowing_doy']), 'harvest_date': doy_to_iso(data['avg_harvest_doy'])} for data in sowing_dates_list}
print('sowing dates loaded')
climate_stations_tif = rasterio.open(os.path.join(settings.BASE_DIR, 'monica', 'monica_geodata', '1000mx1000m', 'nearest_station_per_cultivar', f'nearest_station_cultivar_{cultivar.id}.tif'))
climate_stations_arr = climate_stations_tif.read(1)


print(type(agri_buek_as_array))
# TODO
agri_buek_as_array = agri_buek.read(1)
height, width = agri_buek_as_array.shape
for i in range(0, height):
    for j in range(0, width):
        #print(i, j)
        buek_id = int(agri_buek_as_array[i, j])
        #print(buek_id, buek_id != -9999)
        if  buek_id != -9999:
            print(f"Running cell {i}, {j}")
            buek_id = int(agri_buek_as_array[i, j])
            soil_profile = soil_profile_dict.get(buek_id, None)
            
            station_id = int(climate_stations_arr[i, j])
            print(station_id)
        
        #

Done getting parameters
2734 unique buek ids
Soil profiles loaded
<QuerySet [{'climate_station__id': 3, 'avg_sowing_doy': 278, 'avg_harvest_doy': 215}, {'climate_station__id': 98, 'avg_sowing_doy': 272, 'avg_harvest_doy': 224}, {'climate_station__id': 140, 'avg_sowing_doy': 271, 'avg_harvest_doy': 211}, {'climate_station__id': 164, 'avg_sowing_doy': 264, 'avg_harvest_doy': 201}, {'climate_station__id': 183, 'avg_sowing_doy': 270, 'avg_harvest_doy': 223}, {'climate_station__id': 198, 'avg_sowing_doy': 283, 'avg_harvest_doy': 207}, {'climate_station__id': 222, 'avg_sowing_doy': 276, 'avg_harvest_doy': 223}, {'climate_station__id': 282, 'avg_sowing_doy': 280, 'avg_harvest_doy': 211}, {'climate_station__id': 501, 'avg_sowing_doy': 277, 'avg_harvest_doy': 228}, {'climate_station__id': 524, 'avg_sowing_doy': 288, 'avg_harvest_doy': 220}, {'climate_station__id': 591, 'avg_sowing_doy': 269, 'avg_harvest_doy': 221}, {'climate_station__id': 596, 'avg_sowing_doy': 268, 'avg_harvest_doy': 218}, {'

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [56]:
agri_buek = rasterio.open(os.path.join(settings.BASE_DIR, 'monica', 'monica_geodata', '1000mx1000m', 'buek_id_agriculture_masked_4326.tif'))


In [57]:
agri_buek.read(1).shape

(866, 654)

In [58]:
print(agri_buek.name)

/app/monica/monica_geodata/1000mx1000m/buek_id_agriculture_masked_4326.tif


In [55]:
arr = agri_buek.read(1, masked=False)
print(np.unique(arr))

[-9999    14    25 ... 20170 20171 20173]


In [ ]:
agri_buek.xy(100, 100)[1]

In [10]:
with rasterio.open(os.path.join(settings.BASE_DIR, 'monica',  'nearest_station_cultivar_1.tif')) as f:
    rast = f.read(1)


In [11]:
np.unique(rast)

array([-9999,   140,   183,   591,   596,   640,   662,   717,   821,
         822,   853,   880,   882,   891,  1133,  1412,  1532,  1544,
        1612,  1684,  1766,  1780,  1820,  1957,  2163,  2468,  2625,
        2895,  2995,  3015,  3023,  3028,  3126,  3196,  3287,  3302,
        3314,  3347,  3552,  3637,  3660,  3684,  3750,  3761,  3811,
        4372,  4466,  4642,  4887,  4933,  5100,  5142,  5397,  5404,
        5546,  5750,  5856,  7504,  7532,  7539,  7559,  7564,  7566,
        7592,  7601,  7605,  7608,  7616,  7623,  7624,  7635,  7639,
        7642,  7646,  7650,  7651,  7655,  7662,  7676,  7682,  7697,
        7700,  7711,  7721,  7730,  7736,  7737,  7741,  7743,  7748,
        7756,  7759,  7761,  7770,  7772,  7775,  7786,  7795,  7798,
        7799,  7803,  7807,  7812,  7815,  7824,  7844,  7849,  7857,
        7859,  7862,  7865,  7868,  7872,  7877,  7881,  7883,  7890,
        7897,  7910,  7914,  7920,  7922,  7923,  7931,  7936,  7943,
        7946,  7951,